In [23]:
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
from itertools import combinations
from collections import defaultdict, Counter
#import igraph as ig
import pandas as pd
import sqlite3
import random

env: NX_CUGRAPH_AUTOCONFIG=True


ImportError: cannot import name 'ArrayLikeT' from 'pandas._typing' (/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/pandas/_typing.py)

In [3]:
conn = sqlite3.connect("tiktok_breadth_first.db")
cursor = conn.cursor()

In [4]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'kamalahq'
"""

kamalahq_docs = cursor.execute(query).fetchall()

In [16]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'teamtrump'
"""

teamtrump_docs = cursor.execute(query).fetchall()

In [5]:
kamalahq_token = Counter()
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    kamalahq_token.update(contents)

In [25]:
teamtrump_token = Counter()
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    teamtrump_token.update(contents)

In [27]:
kamalahq_edge_weights = defaultdict(int)
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        kamalahq_edge_weights[(u, v)] += 1

In [28]:
teamtrump_edge_weights = defaultdict(int)
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        teamtrump_edge_weights[(u, v)] += 1

In [ ]:
G_k = nx.Graph()
for (u, v), weight in kamalahq_edge_weights.items():
    G_k.add_edge(u, v, weight=weight)

In [29]:
G_t = nx.Graph()
for (u, v), weight in teamtrump_edge_weights.items():
    G_t.add_edge(u, v, weight=weight)

In [10]:
nx.write_weighted_edgelist(G_k, 'kamalahq_hashtag_network.edgelist')

In [ ]:
nx.write_weighted_edgelist(G_t, 'teamtrump_hashtag_network.edgelist')

In [4]:
G_k = nx.read_weighted_edgelist('kamalahq_hashtag_network.edgelist')

In [8]:
G_t = nx.read_weighted_edgelist('teamtrump_hashtag_network.edgelist')

In [6]:
random.seed(42)
k_betweenness = nx.betweenness_centrality(G_k, k=50)

In [20]:
import json

In [21]:
k_file_path = "k_betweenness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_betweenness, json_file, indent=4)

In [9]:
random.seed(42)
t_betweenness = nx.betweenness_centrality(G_t, k=50)

In [22]:
t_file_path = "t_betweenness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_betweenness, json_file, indent=4)

In [24]:
random.seed(42)
k_closeness = nx.closeness_centrality(G_k, distance='weight')

KeyboardInterrupt: 

In [ ]:
k_file_path = "k_closeness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_closeness, json_file, indent=4)

In [ ]:
random.seed(42)
t_closeness = nx.closeness_centrality(G_t, distance='weight')

In [ ]:
t_file_path = "t_closeness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_closeness, json_file, indent=4)